# TP1: disease modules in the human interactome

The aim of this tutorial is to demonstrate the use of python and the NetworkX package to perform the analysis of one or two sets of proteins within a biological network.

NetworkX is the main graph package, pandas handles tables, NumPy sampling, SciPy the hypergeometric distribution, and Matplotlib figures.

**Learning goals**
- Build, simplify, and extract the main connected interactome.
- Map disease-associated genes to proteins and audit coverage.
- Test module connectivity, internal edges, and network proximity.
- Expand or connect modules; examine centrality, communities, and cliques.
- Compare exclusive disease sets; use S2B calculations.

## Environment and Data SetUp

**Data**

Create a directory where you will save the data files necessary to run this tutorial.

To run this notebook directly palce the data files under a folder named network-biology-assests

For the protein interaction data, go to http://cicblade.dep.usal.es:8080/APID/init.action

1. Select Homo sapiens at the "Organisms with more than 500 interactions" box
2. Select Level 1 (each interaction supported by 2 or more independent experiments)
3. Select "Yes" at the "If you want to filter out inter-species interactions select "YES" and download again" box
4. Click on the "Get selected INTERACTOME" button
5. store the file "9606_noISI_Q2.txt" on the working directory

For the gene disease association data go to:
 https://www.disgenet.org/static/disgenet_ap1/files/downloads/all_gene_disease_associations.tsv.gz

The file should download automatically.

Unzip the file and move the resulting "all_gene_disease_associations.tsv" to the working directory.

**Environment**

To avoid conflicts with other projects it is advisable to create a virtual dedicated environment with the necessary packages. For that you can use venv or conda and the provided requiements.txt file.

You may run the following code on your terminal to setup a venv environment with the necessary packages.

In [ ]:
# Create a virtual environment for the project
python3.13 -m venv bio_net.venv
# Activate the virtual environment
source bio_net.venv/bin/activate
# Install the required dependencies
pip install -r requirements.txt
# deactivate the virtual environment
deactivate

Before starting we will import all necessary packages and set directory paths

In [1]:
#import json
#import os
#import platform
#import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy

In [2]:
WORK_DIR = Path.cwd()
print(f"Current working directory: {WORK_DIR}")
DATA_DIR = WORK_DIR/"network-biology-assets"
print(f"Data directory: {DATA_DIR}")
print(f"\nContents of {DATA_DIR.name}:")
for item in sorted(DATA_DIR.iterdir()):
    print(item.name)


Current working directory: /Users/sofiaagostinho/Documents/Cheat-Sheet-Everything/Bio_networks
Data directory: /Users/sofiaagostinho/Documents/Cheat-Sheet-Everything/Bio_networks/network-biology-assets

Contents of network-biology-assets:
9606_noISI_Q2.txt
all_gene_disease_associations.tsv
isomorphic_graphs.png
lecture-network.png
network-models.png


## Load and audit the APID and DisGenet Data Tables

intro

Required real-data files:
- `9606_noISI_Q2.txt`: the class's human APID interaction snapshot.
- `all_gene_disease_associations.tsv` (or `.tsv.gz`): legacy DisGeNET format.

In [3]:
apid_path = DATA_DIR / "9606_noISI_Q2.txt"
disgene_path = DATA_DIR / "all_gene_disease_associations.tsv"

In [6]:
apid = pd.read_csv(apid_path, sep="\t", index_col=0)
apid.head(10)

,UniprotID_A,UniprotName_A,GeneName_A,UniprotID_B,UniprotName_B,GeneName_B,ExpEvidences,Methods,Publications,3DStructures,CurationEvents
InteractionID,,,,,,,,,,,
1205000,Q14160,SCRIB_HUMAN,SCRIB,B7Z2Y1,B7Z2Y1_HUMAN,NaN,1.0,1.0,1.0,0.0,3.0
1205001,Q14160,SCRIB_HUMAN,SCRIB,Q14155,ARHG7_HUMAN,ARHGEF7,11.0,8.0,8.0,0.0,20.0
1205002,Q14160,SCRIB_HUMAN,SCRIB,Q7Z628,ARHG8_HUMAN,NET1,2.0,2.0,2.0,0.0,2.0
1205003,P22460,KCNA5_HUMAN,KCNA5,Q14160,SCRIB_HUMAN,SCRIB,1.0,1.0,1.0,0.0,2.0
1205004,Q96DN2,VWCE_HUMAN,VWCE,Q14160,SCRIB_HUMAN,SCRIB,1.0,1.0,1.0,0.0,1.0
1205005,O00429,DNM1L_HUMAN,DNM1L,Q14160,SCRIB_HUMAN,SCRIB,1.0,1.0,1.0,0.0,2.0
1205006,Q99569,PKP4_HUMAN,PKP4,Q14160,SCRIB_HUMAN,SCRIB,5.0,5.0,2.0,0.0,5.0
1205007,Q96DL1,NXPE2_HUMAN,NXPE2,Q14160,SCRIB_HUMAN,SCRIB,1.0,1.0,1.0,0.0,2.0
1205008,Q9UQB3,CTND2_HUMAN,CTNND2,Q14160,SCRIB_HUMAN,SCRIB,2.0,2.0,2.0,0.0,2.0


In [7]:
apid.describe(include="all")

,UniprotID_A,UniprotName_A,GeneName_A,UniprotID_B,UniprotName_B,GeneName_B,ExpEvidences,Methods,Publications,3DStructures,CurationEvents
count,213538,213538,212980,213538,213538,213002,213537.000000,213537.000000,213537.000000,213537.000000,213537.000000
unique,14029,14029,13500,15971,15971,15313,NaN,NaN,NaN,NaN,NaN
top,Q13618,CUL3_HUMAN,CUL3,P42858,HD_HUMAN,HTT,NaN,NaN,NaN,NaN,NaN
freq,989,989,989,616,616,616,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,2.209776,2.223446,1.519779,0.013628,3.213284
std,NaN,NaN,NaN,NaN,NaN,NaN,2.696118,1.197341,2.112676,0.802838,3.913106
min,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,0.000000,1.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,2.000000,1.000000,0.000000,2.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,2.000000,1.000000,0.000000,3.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,2.000000,1.000000,0.000000,3.000000


In [8]:
digenet = pd.read_csv(disgene_path, sep="\t")
digenet.head(10)

,geneId,geneSymbol,DSI,DPI,diseaseId,diseaseName,diseaseType,diseaseClass,diseaseSemanticType,score,EI,YearInitial,YearFinal,NofPmids,NofSnps,source
0,1,A1BG,0.7,0.538,C0001418,Adenocarcinoma,group,C04,Neoplastic Process,0.01,1.0,2008.0,2008.0,1,0,LHGDN
1,1,A1BG,0.7,0.538,C0002736,Amyotrophic Lateral Sclerosis,disease,C18;C10,Disease or Syndrome,0.01,1.0,2008.0,2008.0,1,0,BEFREE
2,1,A1BG,0.7,0.538,C0003578,Apnea,phenotype,C23;C08,Sign or Symptom,0.01,1.0,2017.0,2017.0,1,0,BEFREE
3,1,A1BG,0.7,0.538,C0003864,Arthritis,disease,C05,Disease or Syndrome,0.01,1.0,2019.0,2019.0,1,0,BEFREE
4,1,A1BG,0.7,0.538,C0008373,Cholesteatoma,disease,C17,Disease or Syndrome,0.01,1.0,2020.0,2020.0,1,0,BEFREE
5,1,A1BG,0.7,0.538,C0011849,Diabetes Mellitus,group,C18;C19,Disease or Syndrome,0.01,1.0,2018.0,2018.0,1,0,BEFREE
6,1,A1BG,0.7,0.538,C0013080,Down Syndrome,disease,C16;C10,Disease or Syndrome,0.01,1.0,2011.0,2011.0,1,0,BEFREE
7,1,A1BG,0.7,0.538,C0014544,Epilepsy,disease,C10,Disease or Syndrome,0.01,1.0,2019.0,2019.0,1,0,BEFREE
8,1,A1BG,0.7,0.538,C0015230,Exanthema,phenotype,C17,Sign or Symptom,0.01,1.0,2017.0,2017.0,1,0,BEFREE
9,1,A1BG,0.7,0.538,C0016169,pathologic fistula,phenotype,C23,Anatomical Abnormality,0.01,1.0,2018.0,2018.0,1,0,BEFREE


In [9]:
digenet.describe(include="all")

,geneId,geneSymbol,DSI,DPI,diseaseId,diseaseName,diseaseType,diseaseClass,diseaseSemanticType,score,EI,YearInitial,YearFinal,NofPmids,NofSnps,source
count,1.134942e+06,1134942,1.132358e+06,1.132060e+06,1134942,1134942,1134942,978539,1134942,1.134942e+06,967968.000000,967968.000000,967968.000000,1.134942e+06,1.134942e+06,1134942
unique,NaN,21666,NaN,NaN,30170,30170,3,1106,33,NaN,NaN,NaN,NaN,NaN,NaN,878
top,NaN,TNF,NaN,NaN,C0027651,Neoplasms,disease,C04,Disease or Syndrome,NaN,NaN,NaN,NaN,NaN,NaN,BEFREE
freq,NaN,2724,NaN,NaN,10161,10161,781622,132727,439266,NaN,NaN,NaN,NaN,NaN,NaN,789377
mean,2.437042e+06,NaN,5.416682e-01,6.979658e-01,NaN,NaN,NaN,NaN,NaN,6.318843e-02,0.975242,2011.138019,2013.941110,2.632643e+00,2.407973e-01,NaN
std,1.553331e+07,NaN,1.300706e-01,2.029981e-01,NaN,NaN,NaN,NaN,NaN,9.900556e-02,0.136500,7.649745,6.370832,1.949824e+01,5.058252e+00,NaN
min,1.000000e+00,NaN,2.310000e-01,3.800000e-02,NaN,NaN,NaN,NaN,NaN,1.000000e-02,0.000000,1924.000000,1959.000000,0.000000e+00,0.000000e+00,NaN
25%,3.169000e+03,NaN,4.520000e-01,6.150000e-01,NaN,NaN,NaN,NaN,NaN,1.000000e-02,1.000000,2007.000000,2011.000000,1.000000e+00,0.000000e+00,NaN
50%,6.580000e+03,NaN,5.340000e-01,7.690000e-01,NaN,NaN,NaN,NaN,NaN,2.000000e-02,1.000000,2013.000000,2017.000000,1.000000e+00,0.000000e+00,NaN
75%,2.734200e+04,NaN,6.210000e-01,8.460000e-01,NaN,NaN,NaN,NaN,NaN,1.000000e-01,1.000000,2017.000000,2019.000000,2.000000e+00,0.000000e+00,NaN


## Build and clean the interactome

In [10]:
# create a networkx graph from the source and target nodes
full_interactome=nx.from_pandas_edgelist(apid, source='UniprotID_A', target='UniprotID_B', create_using=nx.MultiGraph())

In [ ]:
# check the number of nodes and edges in the full interactome graph
num_nodes = full_interactome.number_of_nodes()
num_edges = full_interactome.number_of_edges()
print(f"Number of nodes: {num_nodes}")
print(f"Number of edges: {num_edges}")

Number of nodes: 17662
Number of edges: 213538


In [14]:
# Now lets clean the graph by removing self-loops and parallel edges. We will convert the MultiGraph to a simple Graph, which will automatically remove parallel edges, and then we will remove self-loops.
clean_interactome = nx.Graph(full_interactome)
# remove self-loops
clean_interactome.remove_edges_from(nx.selfloop_edges(clean_interactome))

# check the number of nodes and edges in the cleaned interactome graph
num_nodes_clean = clean_interactome.number_of_nodes()
num_edges_clean = clean_interactome.number_of_edges()
print(f"Number of nodes in cleaned interactome: {num_nodes_clean}")
print(f"Number of edges in cleaned interactome: {num_edges_clean}")

Number of nodes in cleaned interactome: 17662
Number of edges in cleaned interactome: 211899


In [15]:
# Now lets check the number of components in the cleaned interactome graph and the number of nodes in each component.
components = list(nx.connected_components(clean_interactome))
print(f"Number of components in cleaned interactome: {len(components)}")
for i, component in enumerate(components):
    print(f"Component {i+1} has {len(component)} nodes.")

Number of components in cleaned interactome: 23
Component 1 has 17633 nodes.
Component 2 has 1 nodes.
Component 3 has 2 nodes.
Component 4 has 2 nodes.
Component 5 has 1 nodes.
Component 6 has 2 nodes.
Component 7 has 1 nodes.
Component 8 has 1 nodes.
Component 9 has 1 nodes.
Component 10 has 1 nodes.
Component 11 has 1 nodes.
Component 12 has 1 nodes.
Component 13 has 1 nodes.
Component 14 has 1 nodes.
Component 15 has 1 nodes.
Component 16 has 1 nodes.
Component 17 has 1 nodes.
Component 18 has 1 nodes.
Component 19 has 1 nodes.
Component 20 has 1 nodes.
Component 21 has 2 nodes.
Component 22 has 2 nodes.
Component 23 has 3 nodes.


In [16]:
# From that we will select the largest componet to represent the working interactome.
main_interactome = clean_interactome.subgraph(max(components, key=len)).copy()
# check the number of nodes and edges in the main interactome graph
num_nodes_main = main_interactome.number_of_nodes()
num_edges_main = main_interactome.number_of_edges()
print(f"Number of nodes in main interactome: {num_nodes_main}")
print(f"Number of edges in main interactome: {num_edges_main}")

Number of nodes in main interactome: 17633
Number of edges in main interactome: 211892


## Select disease related genes, map to UniProt, and map to interactome proteins

In [ ]:
# First we need to define the diseases of interest.
